# Fan-Out / Fan-In (Parallel Nodes)

When multiple nodes produce independent results, there is no reason to run them one after another. Fan-out dispatches to several nodes simultaneously; fan-in waits for all of them to finish and merges the results. Total wall time equals the slowest parallel node, not their sum.

**What you'll build:** A document analysis graph that fans out to three simultaneous analyzers (summary, sentiment, keywords), then fans in to a merge node that assembles a structured report.

**Time:** ~20 min | **Difficulty:** Intermediate

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Register parallel branches with `add_parallel_edges()`
- Set a join point with `add_join_edge()`
- Write a merge node that combines results from all branches
- Use state reducers to accumulate list-typed results safely

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Define State with Reducer Fields

The `reducer` annotation on `analysis_notes` tells the graph to concatenate list values from all parallel branches rather than overwriting with the last writer.

In [ ]:
from __future__ import annotations
import asyncio
from dataclasses import dataclass, field
from typing import Annotated

from synapsekit.graph import StateGraph, reducer
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

@dataclass
class AnalysisState:
    document: str                  # Input text to analyze

    # Each parallel branch writes to its own field — no reducer needed
    summary: str   = ""
    sentiment: str = ""
    keywords: str  = ""

    # The merge node assembles this from the three branch results
    report: str = ""

    # Reducer example: if multiple branches append to the same list,
    # the `reducer` annotation tells the graph to concatenate instead of overwrite
    analysis_notes: Annotated[list[str], reducer(list.__add__)] = field(
        default_factory=list
    )

## Step 2: Implement Parallel Branch Nodes

Three independent nodes run concurrently — each reads from `state.document` and writes to its own field.

In [ ]:
llm = OpenAILLM(model="gpt-4o-mini", config=LLMConfig(temperature=0.3))

async def summarize(state: AnalysisState) -> AnalysisState:
    """Produce a 2\u20133 sentence summary of the document."""
    response = await llm.agenerate(
        f"Summarize the following text in 2\u20133 sentences:\n\n{state.document}"
    )
    state.summary = response.text
    state.analysis_notes = [f"Summary length: {len(response.text)} chars"]
    print(f"[summarize] Done.")
    return state


async def analyze_sentiment(state: AnalysisState) -> AnalysisState:
    """Classify the overall sentiment and provide a confidence score."""
    response = await llm.agenerate(
        f"Analyze the sentiment of the following text. "
        f"Return: sentiment (positive/neutral/negative), confidence (0.0\u20131.0), "
        f"and a one-sentence explanation.\n\n{state.document}"
    )
    state.sentiment = response.text
    state.analysis_notes = [f"Sentiment analyzed."]
    print(f"[sentiment] Done.")
    return state


async def extract_keywords(state: AnalysisState) -> AnalysisState:
    """Extract the 5 most important keywords or phrases."""
    response = await llm.agenerate(
        f"Extract the 5 most important keywords or key phrases from the following text. "
        f"Return them as a comma-separated list.\n\n{state.document}"
    )
    state.keywords = response.text
    state.analysis_notes = [f"Keywords: {response.text[:60]}"]
    print(f"[keywords] Done.")
    return state

## Step 3: Implement the Merge (Fan-In) Node

This node runs only after ALL parallel branches have completed. By this point, all three result fields are populated.

In [ ]:
async def merge_results(state: AnalysisState) -> AnalysisState:
    """Assemble all branch outputs into a single structured report.

    This node runs only after ALL parallel branches have completed.
    By this point, state.summary, state.sentiment, and state.keywords
    are all populated.
    """
    state.report = (
        f"DOCUMENT ANALYSIS REPORT\n"
        f"{'='*40}\n\n"
        f"SUMMARY\n{state.summary}\n\n"
        f"SENTIMENT\n{state.sentiment}\n\n"
        f"KEYWORDS\n{state.keywords}\n\n"
        f"NOTES\n" + "\n".join(f"  - {n}" for n in state.analysis_notes)
    )
    print(f"[merge] Report assembled. {len(state.analysis_notes)} analysis notes.")
    return state

## Step 4: Build the Graph with Parallel Edges

`add_parallel_edges()` dispatches copies of the state to all three nodes concurrently. `add_join_edge()` waits for all three to complete before running `merge_results`.

In [ ]:
def build_graph():
    graph = StateGraph(AnalysisState)

    # One entry node kicks things off
    graph.add_node("summarize",         summarize)
    graph.add_node("analyze_sentiment", analyze_sentiment)
    graph.add_node("extract_keywords",  extract_keywords)
    graph.add_node("merge_results",     merge_results)

    graph.set_entry_point("summarize")   # Fan-out starts here by convention;
                                          # the parallel edges launch the other two

    # add_parallel_edges(source, [targets])
    # All three nodes receive the SAME state snapshot and run concurrently.
    # Their return values are merged field-by-field before merge_results runs.
    graph.add_parallel_edges("summarize", ["analyze_sentiment", "extract_keywords"])

    # add_join_edge([sources], target)
    # merge_results runs only after all three parallel nodes have returned.
    graph.add_join_edge(
        ["summarize", "analyze_sentiment", "extract_keywords"],
        "merge_results"
    )

    return graph.compile()

## Complete Working Example

Analyze a document in parallel and measure the elapsed time.

In [ ]:
import time

DOCUMENT = """
Artificial intelligence is transforming the way scientists conduct research.
Machine learning models can now predict protein structures with near-experimental
accuracy, identify patterns in genomic data at scale, and accelerate drug
discovery by screening millions of compounds in silico. Critics argue that over-
reliance on AI may introduce opaque biases into scientific conclusions, while
proponents point to the speed and scale advantages that would be impossible
with manual methods alone.
"""

async def main():
    compiled = build_graph()
    initial = AnalysisState(document=DOCUMENT.strip())

    t0 = time.perf_counter()
    final = await compiled.arun(initial)
    elapsed = time.perf_counter() - t0

    print(f"\nCompleted in {elapsed:.2f}s\n")
    print(final.report)

asyncio.run(main())

## Next Steps

- **[Subgraph Composition](subgraph-composition.ipynb)** — package a fan-out / fan-in as a reusable subgraph
- **[Conditional Routing](conditional-routing.ipynb)** — branch to one of several paths instead of all simultaneously
- **[Parallel Agent Execution](../multi-agent/parallel-agent-execution.ipynb)** — the same pattern without a graph